# 02 — Model evaluation

Rolling-origin results for every tier. All numbers come from
`python -m src.run_experiments`; this notebook only reads the cache.

In [ ]:
import sys, pathlib, json
sys.path.insert(0, str(pathlib.Path.cwd().parent))

import pandas as pd
import plotly.graph_objects as go
from src import config

pd.set_option("display.width", 240)
R = config.RESULTS_DIR

def table(name):
    p = R / f"{name}.csv"
    return pd.read_csv(p) if p.exists() else None

meta = json.loads((R / "run_metadata.json").read_text())
fc   = pd.read_parquet(R / "forecasts.parquet")
print(f"{meta['n_test_origins']:,} test origins, stride {meta['origin_stride_hours']}h, "
      f"horizon {config.HORIZON}h")
print(f"MASE scale {meta['mase_scale']:,.1f} MW (in-sample {meta['stronger_baseline']} on train)")
print(f"test window: {meta['test_start']} -> {meta['test_end']}")

## Headline

Skill is quoted against the **stronger** of the two seasonal naive baselines. MASE below 1 beats the in-sample naive.

In [ ]:
table("summary_detailed").round(3)

## Error by horizon

One averaged MAE hides how skill decays across the day — and hides where the ranking flips.

In [ ]:
c = table("error_by_horizon_detailed")
fig = go.Figure()
for name, d in c.groupby("model"):
    fig.add_trace(go.Scatter(x=d["h"], y=d["MAE"], name=name, mode="lines+markers"))
fig.update_layout(template="plotly_white", height=460, xaxis_title="Hours ahead",
                  yaxis_title="MAE (MW)", title="Error by horizon", hovermode="x unified")
fig

In [ ]:
piv = c.pivot(index="h", columns="model", values="MAE").round(0)
piv["best"] = piv.idxmin(axis=1)
piv

## Statistical significance

A lower MAE is not evidence by itself. Consecutive 24h windows at a 6h stride share target hours, so their loss differentials are autocorrelated; the Diebold-Mariano test below uses a Newey-West HAC variance estimator and origin-level losses.

In [ ]:
table("significance_dm").round(4)

## Distributional accuracy

CRPS scores the entire predictive distribution. Winkler charges interval width **plus** a penalty for misses, so an absurdly wide interval cannot score well the way raw coverage would let it.

In [ ]:
table("probabilistic").round(3)

### Reliability diagram

Points on the diagonal mean the stated quantiles are honest.

In [ ]:
cal = table("calibration")
fig = go.Figure()
fig.add_trace(go.Scatter(x=[0, 1], y=[0, 1], name="Perfect", line=dict(color="#999", dash="dash")))
for name, d in cal.groupby("model"):
    d = d.sort_values("nominal")
    fig.add_trace(go.Scatter(x=d["nominal"], y=d["empirical"], name=name, mode="lines+markers"))
fig.update_layout(template="plotly_white", height=460, xaxis_title="Nominal quantile",
                  yaxis_title="Empirical fraction below", title="Calibration",
                  hovermode="x unified")
fig

### Does calibration survive the horizon?

In [ ]:
cov = table("coverage_by_horizon")
fig = go.Figure()
fig.add_hline(y=0.8, line_dash="dash", line_color="#999", annotation_text="nominal 80%")
for name, d in cov.groupby("model"):
    fig.add_trace(go.Scatter(x=d["h"], y=d["PICP_80"], name=name, mode="lines+markers"))
fig.update_layout(template="plotly_white", height=420, xaxis_title="Hours ahead",
                  yaxis_title="80% interval coverage", hovermode="x unified")
fig

## Peak demand

The daily peak sizes generating reserve, so it is worth scoring separately from average error.

In [ ]:
table("peak_metrics").round(2)

## Stability across the test period

In [ ]:
m = table("metrics_by_month")
fig = go.Figure()
for name, d in m.groupby("model"):
    fig.add_trace(go.Scatter(x=d["month"], y=d["MAE"], name=name, mode="lines+markers"))
fig.update_layout(template="plotly_white", height=420, xaxis_title="",
                  yaxis_title="MAE (MW)", title="MAE by month", hovermode="x unified")
fig

## Which parts of the load curve are hard?

In [ ]:
tod = table("error_by_time_of_day")
fig = go.Figure()
for name, d in tod.groupby("model"):
    fig.add_trace(go.Scatter(x=d["target_hour"], y=d["MAE"], name=name, mode="lines+markers"))
fig.update_layout(template="plotly_white", height=420, xaxis_title="Target hour of day (IST)",
                  yaxis_title="MAE (MW)", hovermode="x unified")
fig

## A sample forecast window, with the full fan

In [ ]:
model = "timesfm_ctx896" if "timesfm_ctx896" in set(fc["model"]) else sorted(fc["model"])[0]
sub = fc[fc["model"] == model]
origins = sorted(sub["origin"].unique())
w = sub[sub["origin"] == origins[len(origins) // 2]].sort_values("h")

fig = go.Figure()
for lo_q, hi_q, level in reversed(config.INTERVAL_LEVELS):
    lo, hi = f"q{lo_q:g}", f"q{hi_q:g}"
    if lo not in w.columns or w[lo].isna().all():
        continue
    fig.add_trace(go.Scatter(x=w["target_time"], y=w[hi], line=dict(width=0), showlegend=False))
    fig.add_trace(go.Scatter(x=w["target_time"], y=w[lo], fill="tonexty",
                             name=f"P{int(lo_q*100)}-P{int(hi_q*100)}",
                             fillcolor=f"rgba(99,110,250,{0.10 + 0.16*(1-level):.2f})",
                             line=dict(width=0)))
fig.add_trace(go.Scatter(x=w["target_time"], y=w["actual"], name="Actual",
                         line=dict(color="#111", width=2.5)))
fig.add_trace(go.Scatter(x=w["target_time"], y=w["prediction"], name=model,
                         line=dict(color="#EF553B", width=2.5, dash="dash")))
fig.update_layout(template="plotly_white", height=460, yaxis_title="Demand (MW)",
                  title=f"{model}", hovermode="x unified")
fig